# Baseline Clean Pipeline

Notebook này chạy baseline theo protocol gần paper hơn nhưng vẫn giữ pipeline hiện tại.

Protocol mới:
- Train: FF++ train only
- Valid: FF++ valid split tách từ FF++ train theo video
- Test-1: FF++ test
- Test-2: Celeb-DF test (OOD)

Eval chạy theo video-style:
- FF++ valid/test: tối đa 20 frames mỗi video
- Celeb-DF test: tối đa 100 frames mỗi video


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = "/content/drive/MyDrive/cs415/moe-deepfake"
REPO_URL = "https://github.com/kdnehihi/SBI-MoE-Deepfake-Detection.git"
PROJECT_DIR = "/content/SBI-MoE-Deepfake-Detection"


In [ ]:
%cd /content
!rm -rf "$PROJECT_DIR"
!git clone "$REPO_URL" "$PROJECT_DIR"
%cd {PROJECT_DIR}


## 1. Check processed data on Drive

Notebook này **không extract lại frame**. Nó dùng luôn:
- `data/processed/ffpp_generalization`
- `data/processed/celebdf`

đã có sẵn trên Drive.


In [ ]:
!ls "$DATA_ROOT/data/processed"
!ls -lh "$DATA_ROOT/data/processed/ffpp_generalization"/*.jsonl
!ls -lh "$DATA_ROOT/data/processed/celebdf"/*.jsonl


## 2. Build paper-like baseline dataset


In [ ]:
!python data/prepare_baseline_clean.py \
  --celebdf-root "$DATA_ROOT/data/processed/celebdf" \
  --ffpp-root "$DATA_ROOT/data/processed/ffpp_generalization" \
  --output-root "$DATA_ROOT/data/baseline" \
  --val-ratio 0.125 \
  --overwrite


In [ ]:
!ls -lh "$DATA_ROOT/data/baseline"/*.jsonl
!wc -l "$DATA_ROOT/data/baseline/train_manifest.jsonl" "$DATA_ROOT/data/baseline/val_manifest.jsonl" "$DATA_ROOT/data/baseline/test_ffpp_manifest.jsonl" "$DATA_ROOT/data/baseline/test_celebdf_manifest.jsonl"


In [ ]:
!echo "train_manifest.jsonl" && head -n 2 "$DATA_ROOT/data/baseline/train_manifest.jsonl"
!echo "val_manifest.jsonl" && head -n 2 "$DATA_ROOT/data/baseline/val_manifest.jsonl"
!echo "test_ffpp_manifest.jsonl" && head -n 2 "$DATA_ROOT/data/baseline/test_ffpp_manifest.jsonl"
!echo "test_celebdf_manifest.jsonl" && head -n 2 "$DATA_ROOT/data/baseline/test_celebdf_manifest.jsonl"


## 3. Train baseline

Paper gốc dùng gần kiểu:
- batch size 32
- 20 epochs
- FF++ valid theo video
- Celeb-DF test theo video

Nếu Colab thiếu VRAM, hạ `--batch-size` xuống 16.


In [ ]:
!python -u train_baseline.py \
  --dataset-root "$DATA_ROOT/data/baseline" \
  --output-dir "$DATA_ROOT/outputs" \
  --batch-size 32 \
  --epochs 20 \
  --num-workers 4 \
  --image-size 224 \
  --device cuda
